In [1]:
import random
import sqlite3

# 连接到Chinook数据库
conn = sqlite3.connect('chinook.db')

# 创建一个游标对象
cursor = conn.cursor()

# 获取数据库中所有表的名称
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
tables

[('albums',),
 ('sqlite_sequence',),
 ('artists',),
 ('customers',),
 ('employees',),
 ('genres',),
 ('invoices',),
 ('invoice_items',),
 ('media_types',),
 ('playlists',),
 ('playlist_track',),
 ('tracks',),
 ('sqlite_stat1',)]

## 数据库解析

In [2]:
'''数据库解析'''
from typing import Union
import traceback
from sqlalchemy import create_engine, inspect, func, select, Table, MetaData, text
import pandas as pd

class DBParser:
    '''DBParser'''
    def __init__(self, db_url:str) -> None:
        '''初始化
        db_url: 数据库链接地址
        '''

        # 判断数据库类型
        if 'sqlite' in db_url:
            self.db_type = 'sqlite'
        elif 'mysql' in db_url:
            self.db_type = 'mysql'

        # 链接数据库
        self.engine = create_engine(db_url, echo=False)
        self.conn = self.engine.connect()
        self.db_url = db_url

        # 查看表名
        self.inspector = inspect(self.engine)
        self.table_names = self.inspector.get_table_names()

        self._table_fields = {} # 数据表字段
        self._table_sample = {} # 数据表样例
        # 依次对每张表的字段进行统计
        for table_name in self.table_names:
            print("Table ->", table_name)
            self._table_fields[table_name] = {}

            # 获取当前表的字段信息
            table_instance = Table(table_name, MetaData(), autoload_with=self.engine)
            table_columns = self.inspector.get_columns(table_name)
            self._table_fields[table_name] = {x['name']:x for x in table_columns}

            # 对当前字段进行统计
            for column_meta in table_columns:
                # 获取当前字段
                column_instance = getattr(table_instance.columns, column_meta['name'])

                # 统计unique
                query = select(func.count(func.distinct(column_instance)))
                distinct_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['distinct'] = distinct_count


            # 获取表样例（第一行）
            query = select(table_instance)
            self._table_sample[table_name] = pd.DataFrame([self.conn.execute(query).fetchone()])
    def get_table_fields(self, table_name) -> pd.DataFrame:
        '''获取表字段信息'''
        return pd.DataFrame.from_dict(self._table_fields[table_name]).T

    def get_table_sample(self, table_name) -> pd.DataFrame:
        '''获取数据表样例'''
        return self._table_sample[table_name]

    def check_sql(self, sql) -> Union[bool, str]:
        '''检查sql是否合理
        参数
            sql: 待执行句子

        返回: 是否可以运行 报错信息
        '''
        try:
            with self.engine.connect() as conn:
                conn.execute(text(sql))
                conn.commit()  # 对于查询非必需，但安全
            return True, 'ok'
        except:
            err_msg = traceback.format_exc()
            return False, err_msg

    def execute_sql(self, sql) -> bool:
        '''运行SQL'''
    
        with self.engine.connect() as conn:
            result = conn.execute(text(sql))
            # 如果需要返回所有行
            return [list(row) for row in result]

In [3]:
parser = DBParser('sqlite:///./chinook.db')

Table -> albums
Table -> artists
Table -> customers
Table -> employees
Table -> genres
Table -> invoice_items
Table -> invoices
Table -> media_types
Table -> playlist_track
Table -> playlists
Table -> tracks


In [4]:
sample = parser.get_table_sample("employees")

In [5]:
parser.get_table_fields("employees")

,name,type,nullable,default,primary_key,distinct
EmployeeId,EmployeeId,INTEGER,False,None,1,8
LastName,LastName,NVARCHAR(20),False,None,0,8
FirstName,FirstName,NVARCHAR(20),False,None,0,8
Title,Title,NVARCHAR(30),True,None,0,5
ReportsTo,ReportsTo,INTEGER,True,None,0,3
BirthDate,BirthDate,DATETIME,True,None,0,8
HireDate,HireDate,DATETIME,True,None,0,7
Address,Address,NVARCHAR(70),True,None,0,8
City,City,NVARCHAR(40),True,None,0,3
State,State,NVARCHAR(40),True,None,0,1


## 模拟生成问题

In [6]:
import time
import jwt
import requests
from itertools import combinations
import numpy as np
from tqdm import tqdm

# 实际KEY，过期时间
def generate_token(apikey: str, exp_seconds: int):
    try:
        id, secret = apikey.split(".")
    except Exception as e:
        raise Exception("invalid apikey", e)

    payload = {
        "api_key": id,
        "exp": int(round(time.time() * 1000)) + exp_seconds * 1000,
        "timestamp": int(round(time.time() * 1000)),
    }
    return jwt.encode(
        payload,
        secret,
        algorithm="HS256",
        headers={"alg": "HS256", "sign_type": "SIGN"},
    )

def ask_glm(question, nretry=5):
    if nretry == 0:
        return None

    url = "https://open.bigmodel.cn/api/paas/v4/chat/completions"
    headers = {
      'Content-Type': 'application/json',
      'Authorization': generate_token("85dbf448eef244a789f6b13a3eaa98f6.DkOIFZloBzStXUbF", 1000)
    }
    data = {
        "model": "glm-3-turbo",
        "p": 0.5,
        "messages": [{"role": "user", "content": question}]
    }
    try:
        response = requests.post(url, headers=headers, json=data, timeout=10)
        return response.json()
    except:
        return ask_glm(question, nretry-1)

In [7]:
answer_prompt = '''你是一个专业的数据库专家，现在需要你结合数据库类型、表的信息和提问，生成对应的SQL语句。请直接输出SQL，不需要有其他输出：

数据库类型：{db_type}

全部表：{tables}

提问：{question}
'''

answer_rewrite_prompt = '''你是一个专业的数据库专家，将下面的问题回答组织为自然语言。：

原始问题：{question}

执行SQL：{sql}

原始结果：{answer}
'''


In [8]:
gt_qes_answer = []

tables = parser.table_names
questions = ["数据库中总共多少个表？","员工表中有多少条记录？","在数据库中所有客户个数和员工个数分别是多少？"]
for question in questions:
    # 生成提问
    try:
        # 生成答案SQL
        input_str = answer_prompt.format(db_type = parser.db_type, tables=tables, question=question)
        sql = ask_glm(input_str)['choices'][0]['message']['content']
        sql = sql.strip('`').strip('\n').replace('sql\n', '')
        sql_answer = ''
        #多条sql
        sqls =sql.split('\n')
        if(len(sqls) > 1):
            sql_answers = []
            for s in sqls:
                # 判断SQL是否符合逻辑
                flag, _ = parser.check_sql(s)
                if not flag:
                    continue
        
                # 获取SQL答案
                sql_answer = parser.execute_sql(s)
                sql_answer = sql_answer[0]
                sql_answers.append(' '.join([str(x) for x in sql_answer]))
            sql_answer = "\n".join(sql_answers)
        else:
            # 判断SQL是否符合逻辑
            flag, _ = parser.check_sql(sql)
            if not flag:
                continue
    
            # 获取SQL答案
            sql_answer = parser.execute_sql(sql)
            sql_answer = sql_answer[0]
            sql_answer = ' '.join([str(x) for x in sql_answer])
            
        # 将SQL和结果改为为自然语言
        input_str = answer_rewrite_prompt.format(question=question, sql=sql, answer=sql_answer)
        nl_answer = ask_glm(input_str)['choices'][0]['message']['content']

        gt_qes_answer.append([
            question, sql, sql_answer, nl_answer
        ])
    except:
        continue

D:\soft\dev\anaconda3\Lib\site-packages\jwt\api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 16 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [10]:
from tabulate import tabulate

# 假设你的数据是 gt_qes_answer
headers = ["问题", "SQL", "结果", "自然语言回答"]
print(tabulate(gt_qes_answer, headers=headers, tablefmt="github"))
# print(gt_qes_answer)

| 问题                                         | SQL                                                      | 结果   | 自然语言回答                                                 |
|----------------------------------------------|----------------------------------------------------------|--------|--------------------------------------------------------------|
| 数据库中总共多少个表？                       | SELECT COUNT(*) FROM sqlite_master WHERE type = 'table'; | 13     | 根据执行的SQL查询结果，数据库中目前共有13个表。              |
| 员工表中有多少条记录？                       | SELECT COUNT(*) FROM employees;                          | 8      | 员工表中目前有8条记录。                                      |
| 在数据库中所有客户个数和员工个数分别是多少？ | SELECT COUNT(*) AS customer_count FROM customers;
SELECT COUNT(*) AS employee_count FROM employees;                                                          | 59
8        | 根据您执行的SQL查询结果，数据库中当前共有59位客户和8位员工。 |
